In [ ]:
import os
import sys
from google.colab import drive

# 1. Montar o Drive (Essencial para persistência de dados e código)
drive.mount('/content/drive', force_remount=True)

# ==========================================================
# 2. CONFIGURAÇÕES DO USUÁRIO (ALTERE AQUI)
# ==========================================================
# Nome da pasta do projeto dentro do seu "Meu Drive"
NOME_DO_REPOSITORIO = "UAVs_forest_fires_STE"

# Nome exato do arquivo da sua chave PRIVADA que está na pasta /.ssh/ do Drive
NOME_DA_CHAVE_PRIVADA = "NOME_DO_SEU_ARQUIVO_DE_CHAVE"
# ==========================================================

# Definição automática de caminhos
project_root = f'/content/drive/MyDrive/{NOME_DO_REPOSITORIO}'
ssh_key_path = f'/content/drive/MyDrive/.ssh/{NOME_DA_CHAVE_PRIVADA}'

# 3. Clone Inicial (Executado apenas se a pasta do projeto não existir no Drive)
if not os.path.exists(project_root):
    print(f"🚀 Pasta '{NOME_DO_REPOSITORIO}' não encontrada no Drive. Clonando via SSH...")

    # Configuração temporária de SSH para permitir o clone inicial
    !mkdir -p ~/.ssh
    !cp {ssh_key_path} ~/.ssh/id_rsa
    !chmod 600 ~/.ssh/id_rsa
    !ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts 2>/dev/null

    # Navega para o Drive e clona o repositório
    %cd /content/drive/MyDrive
    !git clone git@github.com:CarlosCraveiro/{NOME_DO_REPOSITORIO}.git
else:
    print(f"✅ Pasta do projeto '{NOME_DO_REPOSITORIO}' já existe no Drive.")

# 4. Escrita do .bashrc (Automação do Terminal Nativo)
# Injeta configurações para que o terminal abra na pasta correta e com SSH ativo
bashrc_content = f"""
# --- UAVs Project Setup ---
cd {project_root}
mkdir -p ~/.ssh
cp {ssh_key_path} ~/.ssh/id_rsa
chmod 600 ~/.ssh/id_rsa
ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts 2>/dev/null
eval $(ssh-agent -s) > /dev/null
ssh-add ~/.ssh/id_rsa 2>/dev/null
echo "🚀 Terminal pronto em: $(pwd)"
echo "💡 Para recarregar o ambiente: source ~/.bashrc"
"""

# Anexa as configurações ao arquivo de inicialização do shell do Colab
with open('/root/.bashrc', 'a') as f:
    f.write(bashrc_content)

# 5. Adicionar a raiz do projeto ao Path do Python
# Isso permite que os notebooks na pasta /notebooks importem módulos da raiz
if project_root not in sys.path:
    sys.path.append(project_root)

# 6. Configuração de Auto-reload para desenvolvimento em módulos .py
try:
    %load_ext autoreload
    %autoreload 2
    print("✅ Módulos e Auto-reload configurados.")
except Exception as e:
    print(f"⚠️ Nota sobre Autoreload: {e}")

# --- Configuração de Limpeza Automática de Notebooks ---
%cd {project_root}
!pip install nbstripout
!nbstripout --install

# Garante que o .gitattributes exista para filtrar os arquivos
if not os.path.exists(".gitattributes"):
    with open(".gitattributes", "w") as f:
        f.write("*.ipynb filter=nbstripout\n")

print("✅ Limpeza automática de notebooks configurada para o Git.")

print("\n✨ Configuração finalizada. Abra o terminal nativo (ícone >_) para começar.")